# hypercorn smoke test

ASGI server supporting HTTP/1, HTTP/2, HTTP/3 and WebSocket.

In [ ]:
# Example 1: build a default Config and inspect canonical fields.
from hypercorn.config import Config

cfg = Config()
print(f"bind = {cfg.bind}")
print(f"workers = {cfg.workers}")
print(f"worker_class = {cfg.worker_class}")
print(f"keep_alive_timeout = {cfg.keep_alive_timeout}")
print(f"h11_max_incomplete_size = {cfg.h11_max_incomplete_size}")

assert isinstance(cfg.bind, list)
assert cfg.workers >= 1
assert cfg.worker_class in {"asyncio", "uvloop", "trio"}

In [ ]:
# Example 2: mutate Config and verify bind / SSL-less defaults round-trip.
from hypercorn.config import Config

cfg = Config()
cfg.bind = ["127.0.0.1:9000", "127.0.0.1:9001"]
cfg.accesslog = "-"
cfg.errorlog = "-"
cfg.loglevel = "INFO"

sockets = cfg.bind
assert sockets == ["127.0.0.1:9000", "127.0.0.1:9001"]
assert cfg.ssl_enabled is False  # no certfile/keyfile set

# Config.from_mapping is the canonical programmatic-config entrypoint.
cfg2 = Config.from_mapping({"bind": ["0.0.0.0:8080"], "workers": 2})
assert cfg2.bind == ["0.0.0.0:8080"]
assert cfg2.workers == 2

In [ ]:
# Example 3: pull in the public middleware surface.
from hypercorn.middleware import (
    AsyncioWSGIMiddleware,
    DispatcherMiddleware,
    HTTPToHTTPSRedirectMiddleware,
    ProxyFixMiddleware,
)

async def app(scope, receive, send):
    pass

wrapped = HTTPToHTTPSRedirectMiddleware(app, host="example.com")
dispatcher = DispatcherMiddleware({"/api": app, "/": app})
proxy = ProxyFixMiddleware(app, mode="legacy", trusted_hops=1)

assert callable(wrapped)
assert callable(dispatcher)
assert callable(proxy)
assert AsyncioWSGIMiddleware is not None

In [ ]:
# Example 4: CLI config loader builds and yields a Config.
from hypercorn.__main__ import _load_config  # internal but stable across 0.x
from hypercorn.config import Config

# The library's own config loader; with no path it returns defaults.
# Signature in hypercorn 0.18: _load_config(config_path: str | None) -> Config
cfg = _load_config(None)
assert isinstance(cfg, Config)
assert cfg.workers >= 1
print(f"default bind   = {cfg.bind}")
print(f"default loglvl = {cfg.loglevel}")

In [ ]:
# Example 5: serve() coroutine is importable and is a coroutine function.
import inspect

from hypercorn.asyncio import serve as asyncio_serve
from hypercorn.config import Config

assert inspect.iscoroutinefunction(asyncio_serve)

# Verify the worker classes the Config string-names resolve to are importable.
from hypercorn.asyncio import worker_serve  # asyncio worker entry
assert inspect.iscoroutinefunction(worker_serve)

cfg = Config()
cfg.bind = ["127.0.0.1:0"]
cfg.workers = 1
assert cfg.bind == ["127.0.0.1:0"]
print("hypercorn asyncio.serve + worker_serve are async-callable")